Download ASI data from 1984 to 2024; using first growing seasion; for cropland

In [ ]:
import os
import urllib.request

# Create a folder to store the downloaded files
output_folder = "asis_data"
os.makedirs(output_folder, exist_ok=True)

# Loop through years and download each file
for i in range(1984, 2024):
    url = f'https://storage.googleapis.com/fao-gismgr-asis-data/DATA/ASIS/MAPSET/ASI-A/ASIS.ASI-A.{str(i)}.GS1.LC-C.tif'
    output_path = os.path.join(output_folder, f'ASIS.ASI-A.{str(i)}.GS1.LC-C.tif')

    try:
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, output_path)
        print(f"Saved to {output_path}")
    except Exception as e:
        print(f"Failed to download {url}: {e}")



Saved to asis_data/ASIS.ASI-A.1984.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1985.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1986.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1987.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1988.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1989.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1990.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1991.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1992.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1993.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1994.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1995.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1996.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1997.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1998.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.1999.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.2000.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.2001.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.2002.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.2003.GS1.LC-C.tif
Saved to asis_data/ASIS.ASI-A.2004.GS1.L

calculate return period layer

In [ ]:
!pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 77.9 MB/s eta 0:00:00


In [ ]:
!pip install tqdm

In [ ]:
# === Full Gumbel-based vectorized raster processing script ===

import os
import numpy as np
import rasterio
from rasterio.windows import Window
from osgeo import gdal
from tqdm.contrib.concurrent import process_map

# === Parameters ===
input_folder = "asis_data"
years = list(range(1984, 2024))
return_periods = [10, 30, 50, 100]
threshold = 30
vrt_path = os.path.join(input_folder, "asis_stack.vrt")

# === Step 1: Build VRT stack ===
tif_paths = [os.path.join(input_folder, f"ASIS.ASI-A.{year}.GS1.LC-C.tif") for year in years]
if not os.path.exists(vrt_path):
    print("🛠️ Building VRT stack...")
    gdal.BuildVRT(vrt_path, tif_paths, separate=True)
else:
    print("✅ VRT stack already exists.")

# === Step 2: Open VRT and get metadata ===
with rasterio.open(vrt_path) as src:
    height, width = src.height, src.width
    count = src.count
    crs = src.crs
    transform = src.transform
    dtype = rasterio.float32
    blockxsize, blockysize = 256, 256

    profile = {
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'count': 1,
        'dtype': dtype,
        'crs': crs,
        'transform': transform,
        'tiled': True,
        'blockxsize': blockxsize,
        'blockysize': blockysize,
        'compress': 'lzw'
    }

    all_windows = [
        Window(col, row, min(blockxsize, width - col), min(blockysize, height - row))
        for row in range(0, height, blockysize)
        for col in range(0, width, blockxsize)
    ]

# Updated vectorized Gumbel fitting function with ASIS data range filter

def fit_gumbel_tile_vectorized(data, return_periods=[100], threshold=30):
    """
    Vectorized Gumbel fitting for a raster tile.
    Accepts ASIS values in [0, 100]; output return levels are clipped to [0, 100].
    """
    years, h, w = data.shape
    data = np.where((data < 0) | (data > 100), np.nan, data)  # filter invalid input

    flat = data.reshape(years, -1)  # (years, pixels)
    valid_counts = (~np.isnan(flat)).sum(axis=0)
    enough_data = valid_counts > 3

    max_vals = np.full(flat.shape[1], np.nan)
    std_vals = np.full(flat.shape[1], np.nan)

    if np.any(enough_data):
        max_vals[enough_data] = np.nanmax(flat[:, enough_data], axis=0)
        std_vals[enough_data] = np.nanstd(flat[:, enough_data], axis=0)

    valid_mask = enough_data & (max_vals >= threshold) & (std_vals > 1e-3)

    result = np.full((flat.shape[1], len(return_periods)), np.nan, dtype=np.float32)

    if np.any(valid_mask):
        valid_data = flat[:, valid_mask]
        gamma = 0.5772
        mean = np.nanmean(valid_data, axis=0)
        std = np.nanstd(valid_data, axis=0)

        loc = mean - gamma * std
        scale = std * np.sqrt(6) / np.pi

        for i, rp in enumerate(return_periods):
            result[valid_mask, i] = loc - scale * np.log(-np.log(1 - 1 / rp))

    # Ensure all return levels are clipped to valid ASIS range
    return np.clip(result.reshape(h, w, len(return_periods)), 0, 100)




# === Tile processor using vectorized Gumbel ===
def process_tile(win):
    with rasterio.open(vrt_path) as src:
        data = src.read(window=win)  # shape: (years, h, w)
    result = fit_gumbel_tile_vectorized(data, return_periods, threshold)
    return (win, result)

# === Step 3: Process all tiles in parallel ===
print(f"🚀 Processing {len(all_windows)} tiles with Gumbel fitting (vectorized)...")
results = process_map(process_tile, all_windows, max_workers=os.cpu_count(), chunksize=1, desc="Processing Tiles")

# === Step 4: Write output raster(s) ===
with rasterio.open(vrt_path) as src:
    profile['dtype'] = rasterio.float32
    for i, rp in enumerate(return_periods):
        out_path = os.path.join(input_folder, f"ASI_return_level_{rp}yr.tif")
        with rasterio.open(out_path, 'w', **profile) as dst:
            for win, result in results:
                dst.write(result[:, :, i], 1, window=win)

print("\n✅ Gumbel-based return period rasters saved!")


🛠️ Building VRT stack...
🚀 Processing 9164 tiles with Gumbel fitting (vectorized)...


Processing Tiles:   0%|          | 0/9164 [00:00<?, ?it/s]


✅ Gumbel-based return period rasters saved!


In [ ]:
!zip -r /content/asis_data.zip /content/asis_data

  adding: content/asis_data/ (stored 0%)
  adding: content/asis_data/ASIS.ASI-A.2011.GS1.LC-C.tif (deflated 15%)
  adding: content/asis_data/ASIS.ASI-A.2005.GS1.LC-C.tif (deflated 16%)
  adding: content/asis_data/ASIS.ASI-A.2007.GS1.LC-C.tif (deflated 14%)
  adding: content/asis_data/ASIS.ASI-A.2014.GS1.LC-C.tif (deflated 15%)
  adding: content/asis_data/ASIS.ASI-A.2002.GS1.LC-C.tif (deflated 15%)
  adding: content/asis_data/ASIS.ASI-A.1987.GS1.LC-C.tif (deflated 15%)
  adding: content/asis_data/ASIS.ASI-A.1988.GS1.LC-C.tif (deflated 16%)
  adding: content/asis_data/ASIS.ASI-A.1999.GS1.LC-C.tif (deflated 17%)
  adding: content/asis_data/ASIS.ASI-A.1996.GS1.LC-C.tif (deflated 16%)
  adding: content/asis_data/ASIS.ASI-A.1997.GS1.LC-C.tif (deflated 16%)
  adding: content/asis_data/ASIS.ASI-A.2019.GS1.LC-C.tif (deflated 14%)
  adding: content/asis_data/ASIS.ASI-A.2000.GS1.LC-C.tif (deflated 16%)
  adding: content/asis_data/ASIS.ASI-A.2003.GS1.LC-C.tif (deflated 14%)
  adding: content/asis_

In [ ]:
import shutil

drive_folder = "/content/drive/MyDrive/CCRI/return_levels"

# Make sure the target folder exists
os.makedirs(drive_folder, exist_ok=True)

# Copy all output rasters
for rp in return_periods:
    src_path = os.path.join(input_folder, f"ASI_return_level_{rp}yr.tif")
    dst_path = os.path.join(drive_folder, f"ASI_return_level_{rp}yr.tif")
    shutil.copy2(src_path, dst_path)
    print(f"✅ Copied {src_path} to {dst_path}")

✅ Copied asis_data/ASI_return_level_10yr.tif to /content/drive/MyDrive/CCRI/return_levels/ASI_return_level_10yr.tif
✅ Copied asis_data/ASI_return_level_30yr.tif to /content/drive/MyDrive/CCRI/return_levels/ASI_return_level_30yr.tif
✅ Copied asis_data/ASI_return_level_50yr.tif to /content/drive/MyDrive/CCRI/return_levels/ASI_return_level_50yr.tif
✅ Copied asis_data/ASI_return_level_100yr.tif to /content/drive/MyDrive/CCRI/return_levels/ASI_return_level_100yr.tif


In [ ]:
# Full code to extract pixel series, fit Gumbel and GEV, and compare return levels safely

import os
import numpy as np
import rasterio
from rasterio.windows import Window
from scipy.stats import genextreme
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import genpareto

# === Parameters ===
vrt_path = "asis_data/asis_stack.vrt"
target_x, target_y = 11.288,3.291  # longitude, latitude
return_periods = [10, 30, 50, 100]

# === Step 1: Extract pixel series safely ===
with rasterio.open(vrt_path) as src:
    row, col = src.index(target_x, target_y)
    pixel_series = [src.read(i, window=Window(col, row, 1, 1))[0, 0] for i in range(1, src.count + 1)]

pixel_series = np.array(pixel_series, dtype=np.float32)
pixel_series = pixel_series[~np.isnan(pixel_series)]

# === Gumbel fitting (method of moments) ===
gamma = 0.5772
mean = np.mean(pixel_series)
std = np.std(pixel_series)
loc_gumbel = mean - gamma * std
scale_gumbel = std * np.sqrt(6) / np.pi
gumbel_return_levels = [loc_gumbel - scale_gumbel * np.log(-np.log(1 - 1 / rp)) for rp in return_periods]

# === GPD fitting (Peaks Over Threshold) ===
threshold = 20
excesses = pixel_series[pixel_series > threshold] - threshold

if len(excesses) > 0:
    c, loc, scale = genpareto.fit(excesses, floc=0)  # Fix location to 0
    n = len(pixel_series)
    nu = len(excesses)
    pu = nu / n
    gpd_return_levels = [threshold + genpareto.ppf(1 - 1 / (rp * pu), c, loc=0, scale=scale) for rp in return_periods]
else:
    gpd_return_levels = [np.nan] * len(return_periods)

# === Combine into comparison DataFrame ===
comparison = pd.DataFrame({
    "Return Period": return_periods,
    "Gumbel RL": np.round(gumbel_return_levels, 2),
    "GPD RL": np.round(gpd_return_levels, 2)
})


